System information (for reproducibility):

In [ ]:
versioninfo()

Load packages:

In [ ]:
using Pkg

Pkg.activate(pwd())
Pkg.instantiate()
Pkg.status()

## Q1. [MUON Optimizer](https://kellerjordan.github.io/posts/muon/)

### Q1.1 (10 pts) Projection to the Stiefel manifold

Given $G \in \mathbb{R}^{m \times n}$, derive an analytical solution $O$, using SVD of $G$, that minimizes $\|G - O\|_F^2$ subject to $O^T O = I$ or $O O^T = I$.

### Q1.2 (15 pts) SVD implementation

Implement the above SVD-based projection in Julia. Test your implementation on a random matrix $G$ and verify that the solution $O$ satisfies the orthogonality constraint.

In [ ]:
function proj_svd(G::AbstractMatrix)
    # TODO: implement the SVD-based projection to the Stiefel manifold
    return O # placeholder, replace with the actual projection
end

In [ ]:
Random.seed!(257)

# dimension of Wk (key matrix) in the Llama 405B training
m, n = 16_384, 1_024
G = randn(m, n)

In [ ]:
O_svd = proj_svd(G)
println("Orthogonality check: ", norm(O_svd' * O_svd - I))

### Q1.3 (10 pts) Newton-Schulz iteration

Consider the Newton-Schulz iteration
$$
\begin{align*}
&G \gets G / \|G\|_F \\
&\text{for iter in 1:maxiter} \\
&\qquad G \gets a G + b G G^T G + c (G G^T)^2 G \\
&\text{end}
\end{align*}
$$
Explain why, for the choice $(a, b, c) = (1.5, -0.5, 0)$, the iteration converges to the projection of $G$ onto the Stiefel manifold.



### Q1.4 (15 pts) NS implementation

Implement the above Newton-Schulz iteration in Julia. Test your implementation on the same random matrix $G$ and verify that the solution converges to the same projection as the SVD-based method.

In [ ]:
function proj_ns(
    G :: AbstractMatrix{T}, 
    a, 
    b, 
    c;
    maxiter      = 10,
    storage_mn   = similar(G),
    storage_nn_1 = similar(G, size(G, 2), size(G, 2)),
    storage_nn_2 = similar(G, size(G, 2), size(G, 2)),
    O = similar(G)
    ) where T
    # scale singular values to [0, 1]
    O .= G ./ norm(G)
    for iter in 1:maxiter
        # TODO: implement the Newton-Schulz iteration
    end
    return O
end

In [ ]:
O_ns = proj_ns(G, 2, -1.5, 0.5, maxiter=10)
# check whether the solution is close to the SVD-based projection
norm(O_ns - O_svd)

### Q1.4 (20 pts) Benchmark

Benchmark the runtime of the SVD-based projection and the Newton-Schulz iteration for different data types (`Float64`, `Float32`, `Float16`, and `BFloat16` if your hardware supports it) and different devices (CPU and GPU if available). Discuss the results and trade-offs between the two methods in terms of accuracy and computational efficiency.

## Q2. Introduction to optimal design

In this exercise, we practice using disciplined convex programming (SDP in particular) to solve optimal design problems.

Consider a linear model
\begin{eqnarray*}
	y_i = \mathbf{x}_i^T \boldsymbol{\beta} + \epsilon_i, \quad i = 1,\ldots, n,
\end{eqnarray*}
where $\epsilon_i$ are independent Gaussian noises with common variance $\sigma^2$. It is well known that the least squares estimate $\hat{\boldsymbol{\beta}}$ is unbiased and has covariance $\sigma^2 (\sum_{i=1}^n \mathbf{x}_i \mathbf{x}_i^T)^{-1}$. 

In **exact optimal design**, given total number of $n$ allowable experiments, we want to choose among a list of $m$ candidate design points $\{\mathbf{x}_1, \ldots, \mathbf{x}_m\}$ such that the covariance matrix is minimized in some sense. In mathematical terms, we want to find an integer vector $\mathbf{n} = (n_1, \ldots, n_m)$ such that $n_i \ge 0$, $\sum_{i=1}^m n_i = n$, and the matrix $\mathbf{V} = \left( \sum_{i=1}^m n_i \mathbf{x}_i \mathbf{x}_i^T \right)^{-1}$ is "small."

In **approximate optimal design**,  we want to find a probability vector $\mathbf{p} = (p_1, \ldots, p_m)$ such that $p_i \ge 0$, $\sum_{i=1}^m p_i = 1$, and the matrix $\mathbf{V} = \left( \sum_{i=1}^m p_i \mathbf{x}_i \mathbf{x}_i^T \right)^{-1}$ is "small."

Commonly used optimal design criteria include:

- In **$D$-optimal design**, we minimize the determinant of $\mathbf{V}$
\begin{eqnarray*}
	&\text{minimize}& \det \left( \sum_{i=1}^m p_i \mathbf{x}_i \mathbf{x}_i^T \right)^{-1} \\
	&\text{subject to}& p_i \ge 0, \sum_{i=1}^m p_i = 1.
\end{eqnarray*}

- In **$E$-optimal design**, we minimize the spectral norm, i.e., the maximum eigenvalue of $\mathbf{V}$
\begin{eqnarray*}
	&\text{minimize}& \lambda_{\text{max}} \left( \sum_{i=1}^m p_i \mathbf{x}_i \mathbf{x}_i^T \right)^{-1} \\
	&\text{subject to}& p_i \ge 0, \sum_{i=1}^m p_i = 1.	
\end{eqnarray*}
Statistically we are minimizing the maximum variance of $\sum_{j=1}^p a_j \text{var}(\hat \beta_j)$ over all vectors $\mathbf{a}$ with unit norm.

- In **$A$-optimal design**, we minimize the trace of $\mathbf{V}$
\begin{eqnarray*}
	&\text{minimize}& \text{tr} \left( \sum_{i=1}^m p_i \mathbf{x}_i \mathbf{x}_i^T \right)^{-1} \\
	&\text{subject to}& p_i \ge 0, \sum_{i=1}^m p_i = 1.
\end{eqnarray*}
Statistically we are minimizing the total variance $\sum_{j=1}^p \text{var}(\hat \beta_j)$.

### Q2.1 (10 pts) 3x4 factorial design

A drug company asks you to help design a two factor clinical trial, in which treatment A has three levels (A1, A2, and A3) and treatment B has four levels (B1, B2, B3, and B4). Drug company also tells you that the treatment combination A3:B4 has undesirable side effects so we ignore this design point. 

Using dummy coding with A1 and B1 as the baseline levels, find the matrix $C$ with each row a unique design point. Please include the intercept term in the design matrix.

### Q2.2 (30 pts) Find approximate optimal designs

Using semidefinite programming (SDP) software to find the approximate D-, E-, and A-optimal designs for this clinical trial.

Hint: This is what I got (using Hypatia solver), which may or may not be correct.

```
Approximate Optimal Design
┌───────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┐
│ design_pt │   D_opt │   E_opt │   A_opt │ D_opt_n │ E_opt_n │ A_opt_n │
│    String │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │
├───────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┤
│      A1B1 │   0.082 │   0.271 │   0.200 │   8.000 │  27.000 │  20.000 │
│      A2B1 │   0.082 │   0.150 │   0.101 │   8.000 │  15.000 │  10.000 │
│      A3B1 │   0.097 │   0.118 │   0.104 │  10.000 │  12.000 │  10.000 │
│      A1B2 │   0.082 │   0.058 │   0.086 │   8.000 │   6.000 │   9.000 │
│      A2B2 │   0.082 │   0.037 │   0.051 │   8.000 │   4.000 │   5.000 │
│      A3B2 │   0.097 │   0.059 │   0.068 │  10.000 │   6.000 │   7.000 │
│      A1B3 │   0.082 │   0.058 │   0.086 │   8.000 │   6.000 │   9.000 │
│      A2B3 │   0.082 │   0.037 │   0.051 │   8.000 │   3.000 │   5.000 │
│      A3B3 │   0.097 │   0.059 │   0.068 │  10.000 │   6.000 │   7.000 │
│      A1B4 │   0.109 │   0.079 │   0.106 │  11.000 │   8.000 │  10.000 │
│      A2B4 │   0.109 │   0.075 │   0.080 │  11.000 │   7.000 │   8.000 │
│       Obj │   8.987 │  13.000 │  38.925 │   8.988 │  13.028 │  38.946 │
└───────────┴─────────┴─────────┴─────────┴─────────┴─────────┴─────────┘
```

### Q2.3 (30 points) Optimal design with nuisance parameters

Suppose the regression coefficients of linear model $\boldsymbol{\beta}$ is partitioned as $\boldsymbol{\beta} = (\boldsymbol{\beta}_0^T, \boldsymbol{\beta}_1^T)^T$, where $\boldsymbol{\beta}_0$ are nuisance parameters and $\boldsymbol{\beta}_1$ are parameters of primary interest. Given an approximate design $\mathbf{p} = (p_1, \ldots, p_m)$, let the information matrix be partitioned accordingly
$$
\mathbf{I}(\mathbf{p}) = \sum_{i=1}^m p_i \mathbf{x}_i \mathbf{x}_i^T =  \begin{pmatrix}
\mathbf{I}_{00} & \mathbf{I}_{01} \\
\mathbf{I}_{10} & \mathbf{I}_{11}
\end{pmatrix}.
$$
Then the information matrix for $\boldsymbol{\beta}_1$ adjusted for nuisance parameter $\boldsymbol{\beta}_0$ is
$$
\mathbf{I}_{1 \mid 0}(\mathbf{p}) = \mathbf{I}_{11} - \mathbf{I}_{10} \mathbf{I}_{00}^{-1} \mathbf{I}_{01}.
$$

Revisiting the 3x4 factorial design problem in 2.1, suppose the drug company only cares about the estimation of A treatment effects. Find the approximate D-, E-, and A-optimal designs.

Hint: [Schur complement lemma](https://ucla-biostat-216.github.io/2024fall/slides/11-pd/11-pd.html#schur-complement-test-for-positive-definiteness).